# Filter Validity — IndoMathReason
Jalankan di Kaggle dengan **GPU T4 x2** atau **P100**.

**Setup sebelum run:**
1. Upload `clean.jsonl` sebagai Kaggle Dataset (New Dataset → upload file → nama dataset bebas)
2. Add dataset tersebut ke notebook ini via `+ Add Data`
3. Enable GPU accelerator di Settings → Accelerator
4. Run All

In [ ]:
!pip install vllm -q

In [ ]:
import os, subprocess

# Fix: Kaggle T4 tidak punya libcuda.so di lib64 — flashinfer JIT butuh ini
stub = "/usr/local/cuda/lib64/stubs/libcuda.so"
target = "/usr/local/cuda/lib64/libcuda.so"
if os.path.exists(stub) and not os.path.exists(target):
    os.symlink(stub, target)
    print("Symlinked libcuda.so")

# Disable flashinfer sampler (hindari JIT compile yang butuh -lcuda)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
print("VLLM_USE_FLASHINFER_SAMPLER=0")

In [ ]:
import json
import os
from pathlib import Path

# Cari clean.jsonl di semua subfolder /kaggle/input
INPUT_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "clean.jsonl":
            INPUT_PATH = os.path.join(root, f)
            break

if INPUT_PATH is None:
    print("TIDAK DITEMUKAN. File di /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(" ", os.path.join(root, f))
else:
    items = [json.loads(l) for l in open(INPUT_PATH, encoding="utf-8") if l.strip()]
    print(f"Loaded {len(items)} soal dari {INPUT_PATH}")
    print("Keys:", list(items[0].keys()))

In [ ]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
BATCH_SIZE = 64

VALIDITY_PROMPT = """\
Kamu adalah penilai soal matematika. Tentukan apakah soal berikut VALID atau TIDAK VALID.

Soal VALID jika:
1. Soal lengkap dan dapat dipahami dari teks saja (tidak butuh gambar/tabel eksternal)
2. Soal memiliki jawaban numerik atau ekspresi matematika yang pasti
3. Soal ditulis dalam Bahasa Indonesia yang jelas

Soal TIDAK VALID jika:
1. Soal membutuhkan referensi visual yang hilang
2. Soal ambigu atau konteks tidak lengkap
3. Soal bersifat konseptual/definitif tanpa jawaban pasti (mis: "apa yang dimaksud dengan...")
4. Teks bukan soal (strategi, tips, materi, daftar isi)

Jawab hanya dengan satu kata: VALID atau TIDAK_VALID

Soal:
{question}

Jawaban:"""

import subprocess
gpu_count = int(subprocess.check_output("nvidia-smi -L | wc -l", shell=True).decode().strip())
print(f"GPU count: {gpu_count}")

llm = LLM(
    model=MODEL_ID,
    gpu_memory_utilization=0.85,
    dtype="float16",
    max_model_len=2048,
    tensor_parallel_size=gpu_count,
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=10)
print("Model loaded.")

In [ ]:
def parse_validity(text: str) -> bool:
    t = text.strip().upper()
    return "VALID" in t and "TIDAK_VALID" not in t

prompts = [
    VALIDITY_PROMPT.format(question=item.get("soal") or item.get("question", ""))
    for item in items
]

all_outputs = []
for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i:i+BATCH_SIZE]
    outputs = llm.generate(batch, sampling_params)
    all_outputs.extend(outputs)
    print(f"  [{i+len(batch)}/{len(prompts)}] done")

print("Inference selesai.")

In [ ]:
OUTPUT_PATH = "/kaggle/working/after_validity.jsonl"

passed, rejected = [], []
with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for item, output in zip(items, all_outputs):
        verdict = output.outputs[0].text
        if parse_validity(verdict):
            passed.append(item)
            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
        else:
            rejected.append(item)

print(f"Total   : {len(items)}")
print(f"Passed  : {len(passed)} ({len(passed)/len(items)*100:.1f}%)")
print(f"Rejected: {len(rejected)} ({len(rejected)/len(items)*100:.1f}%)")
print(f"Output  : {OUTPUT_PATH}")

In [ ]:
print("=== PASSED (5 sample) ===")
for x in passed[:5]:
    print(" -", x.get("soal", "")[:120])

print("\n=== REJECTED (5 sample) ===")
for x in rejected[:5]:
    print(" -", x.get("soal", "")[:120])

In [ ]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/after_validity.jsonl'))